# Message Trainer

> Fill in a module description here

In [ ]:
#| default_exp trainers.msg_trainer

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch
import os
from fastcore.utils import patch
import torch.nn as nn
import torchvision.utils as vutils
import wandb
from c3jepa_wm.utils.checkpointer import RetrospectiveCheckpointer
import hydra
from pathlib import Path

In [ ]:
#| export
class BaseTrainer:
    def __init__(self, 
                 data_module, 
                 device, 
                 slurm_jobid= None, 
                 lr=1e-4, 
                 epochs=100, 
                 project_name="c3jepa_wm",
                 ckp_dir= "checkpoints",
                 save_dir= "outputs"):
        
        self.data_module = data_module
        
        self.device = device
        
        self.data_module.setup()
        self.train_loader = self.data_module.train_dataloader()
        self.val_loader = self.data_module.val_dataloader()

        self.slurm_jobid = slurm_jobid if slurm_jobid else "default_job"
        self.lr = lr
        self.epochs = epochs
        self.project_name = project_name
        self.ckp_dir = ckp_dir
        self.save_dir = save_dir

    def init_optimizer(self):
        return torch.optim.Adam(self.model.parameters(), lr=self.lr)
    
    def train_epoch(self, epoch):
        raise NotImplementedError("train_epoch method must be implemented by subclasses.")   

    def validate_epoch(self, epoch):
        raise NotImplementedError("validate method must be implemented by subclasses.")
    

## VQ-VAE Trainer

In [ ]:
#| export
class VQVAETrainer(BaseTrainer):

    def __init__(self, data_module, model, device, **kwargs):
        super().__init__(
            data_module= data_module,
            device= device, 
            **kwargs)
        
        self.model = model.to(device)
        self.optimizer = self.init_optimizer()
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', factor=0.5, patience=5
        )
        
        self.save_dir = Path(hydra.utils.to_absolute_path(self.save_dir))
        self.save_dir.mkdir(exist_ok=True, parents=True)

        self.ck_pointer = RetrospectiveCheckpointer(project_name= self.project_name,
                                                    ckp_dir= self.ckp_dir,
                                                    slurm_jobid= self.slurm_jobid,
                                                    agent_id= "vqvae_trainer",
                                                    rank= 0, 
                                                    n_best=3)

        # Create output directories for visual inspection
        os.makedirs(os.path.join(self.save_dir, "Reconstructions"), exist_ok=True)
    
    def train_epoch(self, epoch):
        self.model.train()
        # self.model.vq_layer.training= True
        total_loss = 0.0

        for batch_idx, batch in enumerate(self.train_loader):
            # Manually move data to target device (Lightning did this automatically)
            real_img = batch.to(self.device)

            self.optimizer.zero_grad()

            # Forward pass
            recons, input_img, vq_loss, perplexity = self.model(real_img)

            # Call your model's built-in VQ-VAE loss evaluation function
            loss_dict = self.model.loss_function(
                recons,
                input_img,
                vq_loss,
                perplexity,
            )

            loss = loss_dict["loss"]
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()

            total_loss += loss.item()

            # Log step-level metrics directly to WandB
            wandb.log({f"train_{k}": v.item() for k, v in loss_dict.items()})

        avg_loss = total_loss / len(self.train_loader)
        print(f"Epoch {epoch:02d} | Train Loss: {avg_loss:.4f}")
        return avg_loss

    @torch.no_grad()
    def validate_epoch(self, epoch):
        # self.model.vq_layer.training= False
        self.model.eval()
        total_loss = 0.0

        for batch_idx, batch in enumerate(self.val_loader):
            real_img = batch.to(self.device)

            recons, input_img, vq_loss, perplexity = self.model(real_img)
            loss_dict = self.model.loss_function(
                recons, input_img, vq_loss, perplexity, M_N=1.0, batch_idx=batch_idx
            )

            total_loss += loss_dict["loss"].item()

        avg_loss = total_loss / len(self.val_loader)
        print(f"Epoch {epoch:02d} | Val Loss:   {avg_loss:.4f}")

        # Log epoch-level validation loss
        wandb.log({"val_loss": avg_loss, "perplexity": perplexity.item(), "epoch": epoch,})

        self.sample_and_save_images(self.val_loader, epoch)
        return avg_loss

    @torch.no_grad()
    def sample_and_save_images(self, test_loader, epoch):
        """Replicates on_validation_end by reconstructing a fixed test batch."""
        self.model.eval()

        # Grab the first batch of images from your test/val loader
        test_input = next(iter(test_loader)).to(self.device)

        # Reconstruct images using VQ-VAE decoder
        recons = self.model.generate(test_input)

        # Save grid image to disk
        recons_path = os.path.join(
            self.save_dir, "Reconstructions", f"recons_Epoch_{epoch}.png"
        )
        vutils.save_image(recons.data, recons_path, normalize=True, nrow=8)

        # Upload the reconstructed image grid directly into WandB panel
        wandb.log(
            {"Visual Reconstructions": wandb.Image(recons_path, caption=f"Epoch {epoch}")}
        )


In [ ]:
#| export
@patch
def checkpoint(self: VQVAETrainer, epoch, val_loss):
    checkpoint_state = {
            "epoch": epoch,
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "val_loss": val_loss,
        }
    self.ck_pointer.save_checkpoint(state= checkpoint_state, current_acc= -val_loss, step= epoch)

    

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()